In [3]:
import sys
sys.path.insert(0, '../src/models')
sys.path.insert(0, '../src/evaluation')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import glob
import pickle
import mlflow

import uncertainty

print("All imports successful")

All imports successful


In [4]:
GLOBAL_MODEL_PATH = '../models/global_model.pkl'
PROCESSED_DIR = '../data/processed'

calibration_stats = uncertainty.calibrate(
    global_model_path=GLOBAL_MODEL_PATH,
    processed_dir=PROCESSED_DIR,
    site_encoding=2,
    train_frac=0.70,
)

print("\nCalibration stats:")
for k, v in calibration_stats.items():
    print(f"  {k}: {v}")

Global model loaded from ../models/global_model.pkl
Loaded 8 office001 stations.
  office001_19-102-260-1633: 9359 calibration rows, mean residual = 0.0491
  office001_19-102-260-1634: 9665 calibration rows, mean residual = 0.0333
  office001_19-102-260-1635: 9980 calibration rows, mean residual = 0.0259
  office001_19-102-260-1636: 8759 calibration rows, mean residual = 0.0419
  office001_19-102-260-1637: 7009 calibration rows, mean residual = 0.0200
  office001_19-102-260-1638: 5551 calibration rows, mean residual = 0.0272
  office001_19-102-260-1639: 10369 calibration rows, mean residual = 0.0266
  office001_19-102-260-1640: 10623 calibration rows, mean residual = 0.0528

Calibration complete.
  Total conformity scores: 71,315
  q80 (80% interval half-width): 0.0004 sessions
  q90 (90% interval half-width): 0.0200 sessions

Calibration stats:
  n_conformity_scores: 71315
  q80: 0.0004093373713185828
  q90: 0.020028849715818007
  mean_residual: 0.035524995371068055
  median_residual:

In [5]:
coverage_df = uncertainty.evaluate_coverage(
    global_model_path=GLOBAL_MODEL_PATH,
    processed_dir=PROCESSED_DIR,
    site_encoding=2,
    train_frac=0.70,
)

print("\nCoverage results:")
print(coverage_df.to_string(index=False))

Loaded 8 office001 stations.
  office001_19-102-260-1633: coverage_80=0.992, coverage_90=0.997
  office001_19-102-260-1634: coverage_80=0.969, coverage_90=0.992
  office001_19-102-260-1635: coverage_80=0.986, coverage_90=0.996
  office001_19-102-260-1636: coverage_80=0.987, coverage_90=0.993
  office001_19-102-260-1637: coverage_80=0.992, coverage_90=0.998
  office001_19-102-260-1638: coverage_80=0.832, coverage_90=0.927
  office001_19-102-260-1639: coverage_80=0.766, coverage_90=0.883
  office001_19-102-260-1640: coverage_80=0.743, coverage_90=0.862

Coverage results:
               station_id  eval_rows  coverage_80  coverage_90  gap_80  gap_90
office001_19-102-260-1633       4083       0.9919       0.9971  0.1919  0.0971
office001_19-102-260-1634       4215       0.9692       0.9919  0.1692  0.0919
office001_19-102-260-1635       4350       0.9860       0.9963  0.1860  0.0963
office001_19-102-260-1636       3826       0.9872       0.9932  0.1872  0.0932
office001_19-102-260-1637    

In [6]:
for f in sorted(glob.glob('../data/processed/office001_*.parquet')):
    df = pd.read_parquet(f)
    name = f.split('\\')[-1].replace('.parquet', '')
    print(f"{name}: mean_sessions={df['sessions'].mean():.4f}, "
          f"pct_zero={( df['sessions']==0).mean():.3f}")

office001_19-102-260-1633: mean_sessions=0.1433, pct_zero=0.857
office001_19-102-260-1634: mean_sessions=0.0718, pct_zero=0.929
office001_19-102-260-1635: mean_sessions=0.0630, pct_zero=0.937
office001_19-102-260-1636: mean_sessions=0.0960, pct_zero=0.904
office001_19-102-260-1637: mean_sessions=0.0365, pct_zero=0.964
office001_19-102-260-1638: mean_sessions=0.0734, pct_zero=0.927
office001_19-102-260-1639: mean_sessions=0.1109, pct_zero=0.889
office001_19-102-260-1640: mean_sessions=0.1348, pct_zero=0.866
